# Qwen3-32B on Trainium2: vLLM + native TorchNeuron

Runnable top-to-bottom on a `trn2.48xlarge` Deep Learning Desktop over DCV.

**Status: UNVERIFIED.** No cell here has been executed on hardware. Expect to fix things on first run; see `../PLAN.md` open items O-1..O-6.

**Stack** (Neuron SDK 2.33.0): TorchNeuron 2.14.3.0 Beta, vLLM Neuron plugin 0.29.0.1.2.0 Beta (vLLM 0.29.0), PyTorch 2.14, `neuronx-cc` 2.28.7631.0.

**Order matters.** Sections 1-2 are cheap and fail loudly if the driver stack is wrong — do not skip them to get to serving faster. Section 5 (precompile) must finish before Section 7 (benchmark), or you will be measuring compilation instead of inference.

> Running inside a time-boxed Capacity Block? Push artifacts to EFS/S3 **as you go** (Section 10), not at the end. The instance is reclaimed when the block expires and the root EBS volume goes with it.

## 1. Hardware and driver inventory

Confirm 16 Trainium2 devices and a loaded `aws-neuronx-dkms` driver.

**Open item O-1:** record whether `nc_count` reports **8** (physical cores/device) or **4** (logical at LNC2). The repo's `serve` script derives replica core-groups from this, so if it reports physical the replica math is off by 2x and `NEURON_CORES_PER_DEVICE` must be set explicitly.

In [ ]:
!neuron-ls
print('--- device nodes ---')
!ls -1 /dev/neuron* 2>/dev/null | wc -l
print('--- nc_count (OPEN ITEM O-1: physical 8 or logical 4?) ---')
!neuron-ls --json-output 2>/dev/null | grep -m1 nc_count
print('--- driver ---')
!cat /proc/modules | grep -i neuron || echo 'WARNING: neuron driver not loaded'
print('--- instance type ---')
!curl -s -H "X-aws-ec2-metadata-token: $(curl -sX PUT http://169.254.169.254/latest/api/token -H 'X-aws-ec2-metadata-token-ttl-seconds: 60')" http://169.254.169.254/latest/meta-data/instance-type

### Capacity planning math

This drives every config choice downstream, so compute it explicitly rather than assuming.

In [ ]:
DEVICES = 16            # trn2.48xlarge
CORES_PER_DEVICE = 8    # NeuronCore-v3 per Trainium2 device
HBM_GIB_PER_DEVICE = 96
LNC = 2                 # NEURON_LOGICAL_NC_CONFIG
TP = 8                  # published Qwen3-32B recipe

logical_total = DEVICES * CORES_PER_DEVICE // LNC
devices_per_replica = TP * LNC / CORES_PER_DEVICE
print(f'physical NeuronCores : {DEVICES * CORES_PER_DEVICE}')
print(f'logical  cores @LNC{LNC}: {logical_total}')
print(f'total HBM            : {DEVICES * HBM_GIB_PER_DEVICE} GiB')
print(f'TP{TP} replica uses     : {devices_per_replica:.0f} of {DEVICES} devices')
print(f'max concurrent TP{TP}   : {logical_total // TP} replicas on one instance')

## 2. TorchNeuron smoke test

Run this in a **dedicated venv**, not the serving environment. The vLLM plugin depends on `libtorch-neuronx-lite`, which vendors the TorchNeuron compile components; installing `torch-neuronx` alongside it risks a conflict (PLAN.md R-4).

This isolates "is TorchNeuron working?" from "is vLLM working?" — if Section 2 fails, nothing after it is worth debugging.

In [ ]:
%%bash
set -euo pipefail
VENV=${VENV:-$HOME/venvs/torchneuron}
python3 -m venv "$VENV" 2>/dev/null || true
source "$VENV/bin/activate"
pip install -q --upgrade pip
pip install -q "torch-neuronx==2.14.*" "neuronx-cc==2.*" nki \
  --extra-index-url https://pip.repos.neuron.amazonaws.com \
  --extra-index-url https://download.pytorch.org/whl/cpu
echo "--- installed ---"
pip freeze | grep -iE '^torch|neuron|nki'

In [ ]:
%%bash
source "${VENV:-$HOME/venvs/torchneuron}/bin/activate"
cat > /tmp/tn_smoke.py <<'PY'
import torch, torch_neuronx
print(f"PyTorch      : {torch.__version__}")
print(f"torch-neuronx: {torch_neuronx.__version__}")
assert torch.neuron.is_available(), "No Neuron device available"
print(f"devices      : {torch.neuron.device_count()}")

# 1. eager mode
torch.manual_seed(0)
x = torch.randn(2, 3)
expected = x @ x.T
actual = (x.to("neuron") @ x.to("neuron").T).cpu()
torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-5)
print("eager matmul          : PASS")

# 2. torch.compile -- the same Dynamo path vLLM uses, so a failure here
#    localises the problem to TorchNeuron rather than vLLM.
@torch.compile(backend="neuron")
def fn(a):
    return torch.nn.functional.gelu(a @ a.T)

exp2 = torch.nn.functional.gelu(x @ x.T)
got2 = fn(x.to("neuron")).cpu()
torch.testing.assert_close(got2, exp2, rtol=1e-3, atol=1e-3)
print("torch.compile(neuron) : PASS")
PY
python -I /tmp/tn_smoke.py

## 3. Build or pull the serving image

Primary path layers on the AWS DLC. The from-source image (`containers/Dockerfile.from-source`) is the reference recipe and the basis for the eventual EKS image — **build it outside the capacity block**, since dependency resolution is what eats paid Trainium hours.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)/gen-ai-inference-testing/trn2-qwen3-torchneuron"
DLC=public.ecr.aws/neuron/pytorch-inference-vllm-neuronx:0.29.0.1.2.0-neuronx-py314-sdk2.33.0-ubuntu26.04
docker pull "$DLC"
docker buildx build -t openai_server-vllm_native:neuron -f containers/Dockerfile .
echo '--- platform probe (expect: neuron) ---'
docker run --rm --privileged -v /dev:/dev openai_server-vllm_native:neuron \
  python -c "import vllm; from vllm.platforms import current_platform; print(current_platform.device_name)"

## 4. Download the checkpoint

Pin the revision — benchmark numbers are not comparable across checkpoint revisions (PLAN.md O-3).

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)/gen-ai-inference-testing/trn2-qwen3-torchneuron"
export MODEL_ID=Qwen/Qwen3-32B
export BF16_REVISION=main          # pin to a commit SHA for real runs
export DEST=$HOME/snapshots/Qwen3-32B
bash serve/download_model.sh

## 5. Precompile (do not skip)

First run traces on CPU and compiles to NEFF; the docs default the timeout to 1200s. The cache lives at `$VLLM_CACHE_ROOT/neuron/torch_compile/<fingerprint>/dp_<i>/rank_<r>` and the **fingerprint invalidates on any version, compiler-flag or config change** — so freeze your sweep matrix first and precompile once per config you intend to measure.

`VLLM_NEURON_BACKEND` and `VLLM_CACHE_ROOT` must be set **before Python starts**; setting them in-process has no effect.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)/gen-ai-inference-testing/trn2-qwen3-torchneuron"
mkdir -p "$HOME/cache"
docker run --rm --privileged \
  -v /dev:/dev \
  -v "$HOME/snapshots:/snapshots:rw" \
  -v "$HOME/cache:/cache:rw" \
  -e MODEL_PATH=/snapshots/Qwen3-32B \
  -e VLLM_CACHE_ROOT=/cache \
  -e VLLM_NEURON_BACKEND=neuron_native \
  -e NEURON_LOGICAL_NC_CONFIG=2 \
  --shm-size 32g \
  openai_server-vllm_native:neuron \
  bash /opt/program/serve/precompile.sh

## 6. Serve, then prove it is correct before measuring it

A fast wrong answer is worthless. Health, then a known-answer completion, then 16-way concurrency — the tutorial's own validation sequence.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)/gen-ai-inference-testing/trn2-qwen3-torchneuron"
docker run -d --name qwen3-trn2 --privileged \
  -v /dev:/dev \
  -v "$HOME/snapshots:/snapshots:rw" \
  -v "$HOME/cache:/cache:rw" \
  -e MODEL_PATH=/snapshots/Qwen3-32B \
  -e VLLM_CACHE_ROOT=/cache \
  -e VLLM_NEURON_BACKEND=neuron_native \
  -e NEURON_LOGICAL_NC_CONFIG=2 \
  --shm-size 32g -p 8000:8000 \
  openai_server-vllm_native:neuron \
  bash /opt/program/serve/run_qwen3_32b.sh
echo 'started; following logs until healthy (warm start should be quick)'
for i in $(seq 1 240); do
  if curl -sf http://localhost:8000/health >/dev/null; then echo "healthy after ${i}0s"; break; fi
  sleep 10
done
docker logs --tail 30 qwen3-trn2

In [ ]:
import json, concurrent.futures, urllib.request

BASE = "http://localhost:8000"

def chat(prompt, max_tokens=30):
    body = json.dumps({
        "model": "qwen3-32b",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": 0,
    }).encode()
    req = urllib.request.Request(f"{BASE}/v1/chat/completions", data=body,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)["choices"][0]["message"]["content"]

# Known-answer correctness check
out = chat("The capital of France is")
print("response:", out)
assert "paris" in out.lower(), f"CORRECTNESS FAILURE: {out!r}"
print("correctness: PASS")

# 16-way concurrency: all must return non-empty
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
    results = list(ex.map(lambda i: chat(f"In one sentence, describe the number {i}."), range(16)))
empty = [i for i, r in enumerate(results) if not r.strip()]
assert not empty, f"empty responses at indices {empty}"
print(f"concurrency: PASS ({len(results)}/16 non-empty)")

## 7. SLO sweep — `vllm bench serve` (source of truth)

`--ignore-eos` is mandatory: without it output length depends on the model's stopping behaviour and ITL stops being comparable across cells.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)/gen-ai-inference-testing/trn2-qwen3-torchneuron"
export BASE_URL=http://localhost:8000 MODEL=qwen3-32b
export RESULTS_DIR=./results/$(date -u +%Y%m%dT%H%M%SZ)
# Start narrow to confirm the harness works, then widen:
#   SHAPES="4096:256" CONCURRENCIES="1 4"
bash bench/run_vllm_bench.sh
echo "RESULTS_DIR=$RESULTS_DIR" | tee /tmp/last_results_dir

## 8. Cross-validate with Locust

Second opinion only. Divergence over ~10% on median TTFT or throughput means one harness is mis-instrumented — that is a blocker, not something to average away.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)/gen-ai-inference-testing/trn2-qwen3-torchneuron"
source /tmp/last_results_dir
export MODEL=qwen3-32b INPUT_TOKENS=4096 OUTPUT_TOKENS=256
export LOCUST_SUMMARY_JSON="$RESULTS_DIR/locust_c8_summary.json"
locust -f bench/locustfile.py -H http://localhost:8000 \
  --headless -u 8 -r 1 -t 3m --only-summary \
  --csv "$RESULTS_DIR/locust_c8"
echo '--- compare against the in4096_out256_c8 cell ---'
cat "$LOCUST_SUMMARY_JSON"
python3 -c "import json,sys; d=json.load(open('$RESULTS_DIR/vllmbench_in4096_out256_c8.json')); print({k:d.get(k) for k in ('median_ttft_ms','p99_ttft_ms','median_itl_ms','output_throughput')})"

## 9. Derive the SLO

The deliverable. An SLO without its token shape, concurrency, replica count and dtype attached is not reusable — `parse_results.py` records all of them.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)/gen-ai-inference-testing/trn2-qwen3-torchneuron"
source /tmp/last_results_dir
# --hourly-cost: use the real effective rate. For a Capacity Block that is
# upfront_fee / block_hours, not the on-demand list price.
python3 bench/parse_results.py --results-dir "$RESULTS_DIR" --replicas 1 --hourly-cost 35.76
echo; cat "$RESULTS_DIR/slo_knee.md"

In [ ]:
from IPython.display import Image, display
import pathlib
rd = pathlib.Path(open('/tmp/last_results_dir').read().split('=',1)[1].strip())
png = rd / 'slo_curves.png'
display(Image(filename=str(png))) if png.exists() else print(f'no plot at {png}')

### Multi-replica check

A single-replica number does not predict EKS behaviour, where up to 8 TP8 replicas share one instance. Re-run at 2 and 4 replicas (ports 8000/8001/...) and confirm the per-replica SLO holds under contention.

## 10. Persist artifacts, then tear down

Do this **during** the run, not after. When a Capacity Block expires the instance is reclaimed and the root EBS volume is deleted. EFS (`/home/ubuntu/efs`) survives; EBS does not.

The compile cache is the single most valuable artifact — it is what makes the next run start in minutes instead of ~20 minutes per config.

In [ ]:
%%bash
set -euo pipefail
source /tmp/last_results_dir
S3_BUCKET=${S3_BUCKET:-genai-trn2-489829964455-us-east-2}
EFS=${EFS:-$HOME/efs}

mkdir -p "$EFS/trn2-qwen3"
cp -r "$RESULTS_DIR" "$EFS/trn2-qwen3/" && echo "results -> EFS"
cp -r "$HOME/cache" "$EFS/trn2-qwen3/compile-cache" && echo "compile cache -> EFS"

aws s3 sync "$RESULTS_DIR" "s3://$S3_BUCKET/trn2-qwen3/results/$(basename "$RESULTS_DIR")/"
aws s3 sync "$HOME/cache" "s3://$S3_BUCKET/trn2-qwen3/compile-cache/qwen3-32b-tp8/"
echo 'artifacts persisted'

In [ ]:
!docker stop qwen3-trn2 && docker rm qwen3-trn2 && echo 'server stopped'

## Next

- **Qwen3-VL-32B** — report text-only and image TTFT separately; image encode time lands inside TTFT and makes a combined number meaningless.
- **Qwen3-Embedding-8B** — TTFT/ITL do not apply. The SLO is request latency and embeddings/sec against batch size.
- **EKS** — see PLAN.md Phase 8. The from-source image becomes the production image; the compile cache must be a shared RWX volume or baked in, or per-pod cold compiles make autoscaling unusable.